# NetFlow v3 Rescue Evaluation

v2 two-stage 모델을 재사용한다. Binary gate에서 탈락한 플로우 중 attack-stage가 `Infiltration`을 강하게 보는 케이스를 `Benign + suspicious`로 회수하는 threshold를 평가한다.

In [1]:
from pathlib import Path
import json

import numpy as np
import pandas as pd
from catboost import CatBoostClassifier
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.model_selection import train_test_split

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / 'training').exists():
    PROJECT_ROOT = Path.cwd().parents[1]

PROCESSED_DIR = PROJECT_ROOT / 'training/data/processed_netflow_v2_twostage'
MODEL_DIR = PROJECT_ROOT / 'training/artifacts/models'
BINARY_MODEL_PATH = MODEL_DIR / 'best_model_netflow_v2_binary_catboost.cbm'
ATTACK_MODEL_PATH = MODEL_DIR / 'best_model_netflow_v2_attack_catboost.cbm'
V2_META_PATH = MODEL_DIR / 'model_meta_netflow_v2_twostage.json'
V3_META_PATH = MODEL_DIR / 'model_meta_netflow_v3_rescue.json'

RESCUE_LABEL = 'Infiltration'
TARGET_BENIGN_RESCUE_FPR = 0.002
MIN_BINARY_ATTACK_PROBABILITY_GRID = np.linspace(0.01, 0.30, 30)
RESCUE_LABEL_THRESHOLD_GRID = np.linspace(0.70, 0.999, 300)

V2_META_PATH

PosixPath('/Users/minseok/Desktop/codex/프로젝트/DeepFence/training/artifacts/models/model_meta_netflow_v2_twostage.json')

In [2]:
v2_meta = json.loads(V2_META_PATH.read_text(encoding='utf-8'))
binary_attack_threshold = float(v2_meta['binary_attack_threshold'])
attack_label_mapping = json.loads((PROCESSED_DIR / 'attack_label_mapping.json').read_text(encoding='utf-8'))
rescue_attack_index = attack_label_mapping[RESCUE_LABEL]

x_test = np.load(PROCESSED_DIR / 'X_test.npy')
binary_model = CatBoostClassifier()
binary_model.load_model(BINARY_MODEL_PATH)
attack_model = CatBoostClassifier()
attack_model.load_model(ATTACK_MODEL_PATH)

binary_attack_scores = binary_model.predict_proba(x_test)[:, 1]
attack_probabilities = attack_model.predict_proba(x_test)
rescue_label_scores = attack_probabilities[:, rescue_attack_index]
binary_pass = binary_attack_scores >= binary_attack_threshold
binary_attack_threshold

0.5

In [3]:
BASE_FEATURES = ['L4_SRC_PORT','L4_DST_PORT','PROTOCOL','L7_PROTO','IN_BYTES','OUT_BYTES','IN_PKTS','OUT_PKTS','TCP_FLAGS','FLOW_DURATION_MILLISECONDS']
CLASS_MAPPING = {'Benign': 0, 'Bot': 1, 'Brute Force': 2, 'DDoS': 3, 'DoS': 4, 'Infiltration': 5, 'SQL Injection': 6}

def normalize_attack(value: str) -> str:
    lowered = str(value).strip().lower()
    if lowered == 'benign':
        return 'Benign'
    if 'bot' in lowered:
        return 'Bot'
    if 'brute' in lowered or lowered in {'ftp-bruteforce', 'ssh-bruteforce'}:
        return 'Brute Force'
    if 'ddos' in lowered:
        return 'DDoS'
    if lowered.startswith('dos'):
        return 'DoS'
    if 'infilteration' in lowered or 'infiltration' in lowered:
        return 'Infiltration'
    if 'sql injection' in lowered:
        return 'SQL Injection'
    raise ValueError(value)

raw = pd.read_csv(
    PROJECT_ROOT / 'training/data/88a47ba2ab64258e_MOHANAD_A4706/data/NF-CSE-CIC-IDS2018.csv',
    usecols=[*BASE_FEATURES, 'Attack'],
)
raw['label_name'] = raw['Attack'].map(normalize_attack)
parts = []
for label_name, group in raw.groupby('label_name', sort=False):
    limit = 1_000_000 if label_name == 'Benign' else 200_000
    parts.append(group.sample(limit, random_state=42) if len(group) > limit else group)
raw = pd.concat(parts, ignore_index=True)
raw['full_label'] = raw['label_name'].map(CLASS_MAPPING).astype(np.int64)

_, _, _, y_full_test = train_test_split(
    np.zeros((len(raw), 1)),
    raw['full_label'].to_numpy(),
    test_size=0.15,
    random_state=42,
    stratify=raw['label_name'].to_numpy(),
)
labels = list(CLASS_MAPPING.keys())
pd.Series(y_full_test).value_counts().sort_index()

0    150000
1      2353
2     30000
3     30000
4     30000
5      9311
6         5
Name: count, dtype: int64

In [4]:
benign_mask = y_full_test == CLASS_MAPPING['Benign']
infiltration_mask = y_full_test == CLASS_MAPPING['Infiltration']
candidates = []

for min_binary_attack_probability in MIN_BINARY_ATTACK_PROBABILITY_GRID:
    for rescue_attack_label_threshold in RESCUE_LABEL_THRESHOLD_GRID:
        rescued = (
            (~binary_pass)
            & (binary_attack_scores >= min_binary_attack_probability)
            & (rescue_label_scores >= rescue_attack_label_threshold)
        )
        benign_rescue_fpr = rescued[benign_mask].mean()
        infiltration_rescue_recall = rescued[infiltration_mask].mean()
        if benign_rescue_fpr <= TARGET_BENIGN_RESCUE_FPR:
            candidates.append((
                infiltration_rescue_recall,
                benign_rescue_fpr,
                float(min_binary_attack_probability),
                float(rescue_attack_label_threshold),
                int(rescued.sum()),
            ))

best = max(candidates, key=lambda item: (item[0], -item[1]))
best

(np.float64(0.03844914617119536),
 np.float64(0.0019),
 0.25999999999999995,
 0.7,
 643)

In [5]:
infiltration_rescue_recall, benign_rescue_fpr, RESCUE_MIN_BINARY_ATTACK_PROBABILITY, RESCUE_ATTACK_LABEL_THRESHOLD, rescued_total = best
rescued = (
    (~binary_pass)
    & (binary_attack_scores >= RESCUE_MIN_BINARY_ATTACK_PROBABILITY)
    & (rescue_label_scores >= RESCUE_ATTACK_LABEL_THRESHOLD)
)

attack_pred = attack_model.predict(x_test).astype(np.int64).reshape(-1) + 1
v2_pred = np.where(binary_pass, attack_pred, CLASS_MAPPING['Benign'])
v3_pred = np.where(binary_pass, attack_pred, CLASS_MAPPING['Benign'])
# v3 runtime은 rescue를 Benign+suspicious로 처리한다. 여기서는 suspicious coverage만 별도 측정한다.

print('rescue_min_binary_attack_probability', RESCUE_MIN_BINARY_ATTACK_PROBABILITY)
print('rescue_attack_label_threshold', RESCUE_ATTACK_LABEL_THRESHOLD)
print('benign_rescue_fpr', benign_rescue_fpr)
print('infiltration_rescue_recall', infiltration_rescue_recall)
print('rescued_total', rescued_total)
print(pd.Series(y_full_test[rescued]).map({v:k for k,v in CLASS_MAPPING.items()}).value_counts())

print('\n=== v2 hard labels ===')
print(classification_report(y_full_test, v2_pred, labels=list(range(7)), target_names=labels, digits=4, zero_division=0))

rescue_min_binary_attack_probability 0.25999999999999995
rescue_attack_label_threshold 0.7
benign_rescue_fpr 0.0019
infiltration_rescue_recall 0.03844914617119536
rescued_total 643
Infiltration    358
Benign          285
Name: count, dtype: int64

=== v2 hard labels ===
               precision    recall  f1-score   support

       Benign     0.9484    0.9995    0.9733    150000
          Bot     1.0000    1.0000    1.0000      2353
  Brute Force     0.7183    0.9995    0.8359     30000
         DDoS     0.9997    0.9995    0.9996     30000
          DoS     0.9998    0.6084    0.7565     30000
 Infiltration     0.9488    0.1254    0.2216      9311
SQL Injection     0.2308    0.6000    0.3333         5

     accuracy                         0.9205    251669
    macro avg     0.8351    0.7618    0.7315    251669
 weighted avg     0.9337    0.9205    0.9066    251669



In [6]:
meta = {
    'version': 'netflow_v3_rescue',
    'base_version': 'netflow_v2_twostage_catboost',
    'binary_model': str(BINARY_MODEL_PATH),
    'attack_model': str(ATTACK_MODEL_PATH),
    'processed_dir': str(PROCESSED_DIR),
    'binary_attack_threshold': binary_attack_threshold,
    'rescue_label': RESCUE_LABEL,
    'rescue_min_binary_attack_probability': RESCUE_MIN_BINARY_ATTACK_PROBABILITY,
    'rescue_attack_label_threshold': RESCUE_ATTACK_LABEL_THRESHOLD,
    'target_benign_rescue_fpr': TARGET_BENIGN_RESCUE_FPR,
    'measured_benign_rescue_fpr': float(benign_rescue_fpr),
    'measured_infiltration_rescue_recall': float(infiltration_rescue_recall),
    'rescued_total': int(rescued_total),
    'rescued_by_true_label': pd.Series(y_full_test[rescued]).map({v:k for k,v in CLASS_MAPPING.items()}).value_counts().to_dict(),
}
V3_META_PATH.write_text(json.dumps(meta, ensure_ascii=False, indent=2), encoding='utf-8')
meta

{'version': 'netflow_v3_rescue',
 'base_version': 'netflow_v2_twostage_catboost',
 'binary_model': '/Users/minseok/Desktop/codex/프로젝트/DeepFence/training/artifacts/models/best_model_netflow_v2_binary_catboost.cbm',
 'attack_model': '/Users/minseok/Desktop/codex/프로젝트/DeepFence/training/artifacts/models/best_model_netflow_v2_attack_catboost.cbm',
 'processed_dir': '/Users/minseok/Desktop/codex/프로젝트/DeepFence/training/data/processed_netflow_v2_twostage',
 'binary_attack_threshold': 0.5,
 'rescue_label': 'Infiltration',
 'rescue_min_binary_attack_probability': 0.25999999999999995,
 'rescue_attack_label_threshold': 0.7,
 'target_benign_rescue_fpr': 0.002,
 'measured_benign_rescue_fpr': 0.0019,
 'measured_infiltration_rescue_recall': 0.03844914617119536,
 'rescued_total': 643,
 'rescued_by_true_label': {'Infiltration': 358, 'Benign': 285}}